# Analisis Komprehensif Curah Hujan CHIRPS Seluruh Indonesia
Notebook ini berisi alur analisis klimatologi, kejadian ekstrem ($Rx1day$ & $R0.2mm$), Uji Tren Statistik (Mann-Kendall & Sen's Slope), Visualisasi Spasial Diskret (`BoundaryNorm`) dengan pembatas `Indonesia.geojson`, serta ekspor otomatis peta spasial bulanan berstruktur folder `output_spasial_indonesia/YYYY/YYYY_MM/`.

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import pymannkendall as mk
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = "DejaVu Sans"
plt.rcParams['font.family'] = "sans-serif"

## 1. Auto-Detect Path Dataset CHIRPS & Vektor `Indonesia.geojson`

In [ ]:
possible_paths = [
    "/kaggle/input/datasets/jerismeteo/chirps-kebumen/data/CHIRPS/",
    "data/chirps",
    "data/CHIRPS"
]

data_dir = None
for p in possible_paths:
    if os.path.exists(p):
        data_dir = p
        break

if data_dir:
    print(f"✅ Menggunakan direktori data CHIRPS: {data_dir}")
    nc_files = sorted(glob.glob(os.path.join(data_dir, "**/*.nc"), recursive=True))
    print(f"Total file NetCDF ditemukan: {len(nc_files)}")
else:
    raise FileNotFoundError("❌ Folder data CHIRPS tidak ditemukan!")

# Load GeoJSON Indonesia
geojson_path = "Indonesia.geojson"
if os.path.exists(geojson_path):
    gdf_indo = gpd.read_file(geojson_path)
    print(f"✅ Berhasil memuat pembatas {geojson_path}")
else:
    gdf_indo = None
    print("⚠️ File Indonesia.geojson tidak ditemukan.")

## 2. Memuat Multi-File NetCDF CHIRPS

In [ ]:
ds = xr.open_mfdataset(nc_files, combine='by_coords')
print("--- Informasi Dataset CHIRPS ---")
print(ds)

## 3. Analisis Deskriptif & Klimatologi (Climatology Analysis)
- Ekstraksi Deret Waktu Rata-rata Wilayah (*Areal Mean*)
- Agregasi WMO (SUM): Harian, Bulanan, Tahunan
- Pola Siklus Musim Bulanan

In [ ]:
# Areal Mean Time Series (Rata-rata Spasial)
da_areal_mean = ds['precipitation'].mean(dim=['x', 'y']).to_series()
df_daily = pd.DataFrame({'rainfall': da_areal_mean})
df_daily.index.name = 'DateTime'

# Agregasi WMO (SUM)
df_monthly = df_daily.resample('ME').sum()
df_annual = df_daily.resample('YE').sum()

print("--- Ringkasan Statistik Curah Hujan Harian ---")
print(df_daily.describe())

# Plot Time Series Harian & Bulanan
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)
axes[0].plot(df_daily.index, df_daily['rainfall'], color='navy', linewidth=0.8, alpha=0.8)
axes[0].set_title("Deret Waktu Curah Hujan Harian CHIRPS (Rata-rata Indonesia)", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Curah Hujan (mm/hari)")
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].bar(df_monthly.index, df_monthly['rainfall'], color='royalblue', width=20, alpha=0.85)
axes[1].set_title("Akumulasi Curah Hujan Bulanan CHIRPS", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Curah Hujan (mm/bulan)")
axes[1].set_xlabel("Tahun")
axes[1].grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Klimatologi Bulanan
monthly_climatology = df_monthly.groupby(df_monthly.index.month).mean()

plt.figure(figsize=(10, 5))
bars = plt.bar(monthly_climatology.index, monthly_climatology['rainfall'], color='teal', alpha=0.85, edgecolor='black')
plt.title("Klimatologi Curah Hujan Bulanan CHIRPS (Siklus Musiman)", fontsize=14, fontweight='bold')
plt.xlabel("Bulan", fontsize=12)
plt.ylabel("Curah Hujan (mm/bulan)", fontsize=12)
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des'])
plt.grid(True, axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 5, f'{yval:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Analisis Kejadian Ekstrem (Extreme Indices: $Rx1day$ & $R0.2mm$)
- **$Rx1day$**: Curah hujan harian maksimum (mm/hari)
- **$R0.2mm$ / Hari Hujan**: Frekuensi hari hujan (hujan $\ge 0.2\text{ mm/hari}$)

In [ ]:
# 1. Rx1day Temporal & Spasial
rx1day_spasial = ds['precipitation'].max(dim='time')
rx1day_annual = df_daily['rainfall'].resample('YE').max()

# 2. R0.2mm (Hari Hujan dengan threshold >= 0.2 mm/hari)
threshold = 0.2
rain_days_monthly = (df_daily['rainfall'] >= threshold).resample('ME').sum()
rain_days_spasial = (ds['precipitation'] >= threshold).sum(dim='time')

print("--- Indeks Kejadian Ekstrem ---")
print(f"Rx1day Maksimum Tertinggi: {df_daily['rainfall'].max():.2f} mm/hari pada tanggal {df_daily['rainfall'].idxmax().strftime('%Y-%m-%d')}")
print(f"Rata-rata Hari Hujan (R0.2mm) per bulan: {rain_days_monthly.mean():.1f} hari/bulan")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot Rx1day Tahunan
axes[0].plot(rx1day_annual.index.year, rx1day_annual.values, marker='o', color='crimson', linewidth=1.5)
axes[0].set_title("Indeks Rx1day (Hujan Maksimum 1-Hari Tahunan)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Tahun")
axes[0].set_ylabel("Rx1day (mm/hari)")
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot R0.2mm Bulanan
axes[1].hist(rain_days_monthly, bins=15, color='darkcyan', edgecolor='black', alpha=0.8)
axes[1].set_title("Distribusi Frekuensi Hari Hujan Bulanan (R0.2mm)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Jumlah Hari Hujan per Bulan (hari)")
axes[1].set_ylabel("Frekuensi (Bulan)")
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 5. Analisis Tren Statistik (Mann-Kendall & Sen's Slope)
Pengujian tren kenaikan/penurunan curah hujan jangka panjang menggunakan uji **Mann-Kendall** ($p\text{-value}$) dan **Sen's Slope** ($\text{mm/tahun}$).

In [ ]:
# Uji Mann-Kendall Deret Waktu Bulanan
mk_monthly = mk.original_test(df_monthly['rainfall'])
print("=== Hasil Uji Mann-Kendall (Curah Hujan Bulanan) ===")
print(f"Trend          : {mk_monthly.trend}")
print(f"p-value        : {mk_monthly.p:.4f}")
print(f"Z-statistic    : {mk_monthly.z:.4f}")
print(f"Sen's Slope    : {mk_monthly.slope:.4f} mm/bulan")

# Uji Mann-Kendall Deret Waktu Tahunan
mk_annual = mk.original_test(df_annual['rainfall'])
print("\n=== Hasil Uji Mann-Kendall (Curah Hujan Tahunan) ===")
print(f"Trend          : {mk_annual.trend}")
print(f"p-value        : {mk_annual.p:.4f}")
print(f"Z-statistic    : {mk_annual.z:.4f}")
print(f"Sen's Slope    : {mk_annual.slope:.4f} mm/tahun")

# Plot Tren Sen's Slope Tahunan
plt.figure(figsize=(12, 5))
years = df_annual.index.year.values
values = df_annual['rainfall'].values
plt.plot(years, values, label='Akumulasi Tahunan (Observed)', marker='o', color='navy')

# Garis Regresi Sen's Slope
intercept = mk_annual.intercept
trend_line = intercept + mk_annual.slope * np.arange(len(years))
plt.plot(years, trend_line, label=f"Sen's Slope Trend ({mk_annual.slope:.2f} mm/tahun, p={mk_annual.p:.3f})", color='red', linestyle='--', linewidth=2)

plt.title("Analisis Tren Curah Hujan Tahunan CHIRPS (Uji Mann-Kendall & Sen's Slope)", fontsize=13, fontweight='bold')
plt.xlabel("Tahun")
plt.ylabel("Akumulasi Hujan (mm/tahun)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Visualisasi Spasial 2D Diskret (`BoundaryNorm`) & Pembatas `Indonesia.geojson`
Visualisasi sebaran spasial curah hujan 2D menggunakan colormap diskret ilmiah (`mcolors.BoundaryNorm`) dan overlay pembatas wilayah `Indonesia.geojson`.

In [ ]:
# Fungsi Colormap Diskret Ilmiah
def buat_cmap_diskret_ilmiah(bounds_list, nama_palette='YlGnBu'):
    base_cmap = plt.get_cmap(nama_palette)
    norm = mcolors.BoundaryNorm(boundaries=bounds_list, ncolors=base_cmap.N)
    return base_cmap, norm

# Tentukan interval diskret curah hujan (mm)
interval_hujan = [0, 10, 25, 50, 100, 150, 200, 300, 450, 600, 800]
cmap_ilmiah, norm_ilmiah = buat_cmap_diskret_ilmiah(interval_hujan, 'YlGnBu')

# Hitung Akumulasi Total & Rata-rata Spasial
spatial_total = ds['precipitation'].sum(dim='time')
spatial_mean = ds['precipitation'].mean(dim='time')

fig, axes = plt.subplots(2, 1, figsize=(16, 14))

# 1. Peta Total Akumulasi dengan Discrete BoundaryNorm + Indonesia.geojson
im1 = spatial_total.plot(
    ax=axes[0], 
    cmap=cmap_ilmiah, 
    norm=norm_ilmiah, 
    add_colorbar=False
)
if gdf_indo is not None:
    gdf_indo.boundary.plot(ax=axes[0], color='black', linewidth=0.6, alpha=0.8, label='Batas Wilayah Indonesia')
axes[0].set_title("Peta Total Akumulasi Curah Hujan CHIRPS di Indonesia (Discrete BoundaryNorm)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Bujur (Longitude)", fontsize=11)
axes[0].set_ylabel("Lintang (Latitude)", fontsize=11)

cbar1 = fig.colorbar(im1, ax=axes[0], orientation='horizontal', pad=0.06, shrink=0.8, ticks=interval_hujan)
cbar1.set_label('Total Curah Hujan Akumulasi (mm)', fontsize=11)

# 2. Peta Rata-rata Harian
im2 = spatial_mean.plot(
    ax=axes[1], 
    cmap='Blues', 
    add_colorbar=False
)
if gdf_indo is not None:
    gdf_indo.boundary.plot(ax=axes[1], color='black', linewidth=0.6, alpha=0.8)
axes[1].set_title("Peta Rata-rata Curah Hujan Harian CHIRPS (mm/hari)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Bujur (Longitude)", fontsize=11)
axes[1].set_ylabel("Lintang (Latitude)", fontsize=11)

cbar2 = fig.colorbar(im2, ax=axes[1], orientation='horizontal', pad=0.06, shrink=0.8)
cbar2.set_label('Rata-rata Curah Hujan Harian (mm/hari)', fontsize=11)

plt.tight_layout()
plt.show()

## 7. Ekspor Peta Spasial Bulanan Berstruktur Folder `output_spasial_indonesia/YYYY/YYYY_MM/`
Menghasilkan peta 2D spasial bulanan presisi tinggi dengan `BoundaryNorm` dan pembatas `Indonesia.geojson`, lalu menyimpannya secara otomatis ke dalam folder berhirarki `output_spasial_indonesia/YYYY/YYYY_MM/`.

In [ ]:
base_out_dir = Path("output_spasial_indonesia")
base_out_dir.mkdir(exist_ok=True)

# Ambil daftar tahun yang tersedia di dataset
years = np.unique(ds.time.dt.year.values)
# Untuk demonstrasi cepat, ambil tahun 2026 (atau tahun terakhir jika 2026 belum ada)
target_years = [y for y in years if y >= 2026] if any(y >= 2026 for y in years) else years[-1:]
month_names = ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember']

for y in target_years:
    ds_y = ds.sel(time=str(y))
    ds_y_monthly = ds_y['precipitation'].groupby('time.month').sum()
    months = ds_y_monthly.month.values
    
    for m in months:
        # Buat hirarki folder: output_spasial_indonesia/YYYY/YYYY_MM/
        folder_bulan = base_out_dir / str(y) / f"{y}_{m:02d}"
        folder_bulan.mkdir(parents=True, exist_ok=True)
        
        data_m = ds_y_monthly.sel(month=m)
        m_name = month_names[m-1] if m <= 12 else f"Bulan {m}"
        
        fig, ax = plt.subplots(figsize=(14, 8))
        im = data_m.plot(ax=ax, cmap=cmap_ilmiah, norm=norm_ilmiah, add_colorbar=False)
        if gdf_indo is not None:
            gdf_indo.boundary.plot(ax=ax, color='black', linewidth=0.6, alpha=0.85)
            
        ax.set_title(f"Peta Akumulasi Curah Hujan Bulanan CHIRPS - Indonesia\nBulan: {m:02d} ({m_name}) Tahun: {y}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("Longitude", fontsize=11)
        ax.set_ylabel("Latitude", fontsize=11)
        
        cbar = fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.06, shrink=0.8, ticks=interval_hujan)
        cbar.set_label('Total Curah Hujan Bulanan (mm)', fontsize=11)
        
        png_path = folder_bulan / f"chirps_indonesia_{y}_{m:02d}.png"
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved: {png_path}")

print(f"Seluruh peta spasial bulanan berstruktur folder tersimpan di: {base_out_dir}")